<h1 style="font-size:40px;"> Dataframe API </h1>

![spark](img/esquema2.png)

&nbsp;  
&nbsp;  
&nbsp;  

Spark es un framework muy global y nos puede servir para tratar textos, tablas e incluso imágenes y vídeo. Pero la realidad es que mucha de la información a procesar suele ser información estructuradas (tablas, ya sean en csv, parquet, hive o MySQL) y también información semi-estructurada (JSON y XML).

Para trabajar con esta información en spark existe la librería `DataFrame`, inspirada en SQL y por su puesto en los `DataFrame` de pandas. Esta librería es muy intuitiva, muy utilizada y con ella conseguimos una *performance* mucho más alta que con spark core.

![](img/dataframe.png)
<center>
    https://databricks.com/blog/2015/02/17/introducing-dataframes-in-spark-for-large-scale-data-science.html
</center>

## Primeros pasos con `DataFrame` de spark

In [1]:
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
from pyspark import SparkConf
from pyspark.sql import SparkSession

In [2]:
conf = SparkConf().setAppName("[ICAI] Intro Pyspark")

In [3]:
spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


El punto de entrada para trabajar con la librería de `DataFrame` es directamente `spark`:

In [4]:
readme = spark.read.text("/datos/README.md")

In [5]:
type(readme)

pyspark.sql.dataframe.DataFrame

**NOTA:** No hay que confundir estos `DataFrame` que viven en spark con los `Dataframe` de pandas

In [6]:
type(pd.DataFrame(list(range(10))))

pandas.core.frame.DataFrame

In [7]:
readme.take(1)

[Row(value='# Apache Spark')]

In [8]:
readme.count()

103

Aunque el uso es parecido y serán muy fáciles de usar si conocemos *pandas*:

In [9]:
(
    # Leemos el fichero como `Dataframe`
    spark.read.text("/datos/README.md")
    # Dividimos cada línea por espacios y llamamos a la columna word
    .select(F.split(F.col("value"), r"\s+").alias("word"))
    # Desplegamos cada línea generando más filas (similar al flatMap)
    .withColumn("word", F.explode("word"))
    # Agrupamos y contamos
    .groupBy("word")
    .count()
    # Ordenamos de mayor a menor
    .orderBy(F.desc("count"))
    # Mostramos los 10 primeros resultados
).limit(10).show()

+-----+-----+
| word|count|
+-----+-----+
|     |   47|
|  the|   24|
|   to|   17|
|Spark|   16|
|  for|   12|
|  and|    9|
|   ##|    9|
|    a|    8|
|  can|    7|
|   on|    7|
+-----+-----+



Podemos generar nuevos DataFrame basados en transformaciones de otros DataFrame igual que haciamos con RDD

In [10]:
wordcount = (
    # Leemos el fichero como `Dataframe`
    spark.read.text("/datos/README.md")
    # Dividimos cada línea por espacios y llamamos a la columna word
    .select(F.split(F.col("value"), r"\s+").alias("word"))
    # Desplegamos cada línea generando más filas (similar al flatMap)
    .withColumn("word", F.explode("word"))
    # Agrupamos y contamos
    .groupBy("word")
    .count()
)

Incluso podemos explora el `DAG`:

In [11]:
type(wordcount)

pyspark.sql.dataframe.DataFrame

In [12]:
wordcount.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[word#40], functions=[count(1)])
   +- Exchange hashpartitioning(word#40, 200), ENSURE_REQUIREMENTS, [plan_id=123]
      +- HashAggregate(keys=[word#40], functions=[partial_count(1)])
         +- Generate explode(word#36), false, [word#40]
            +- Project [split(value#34, \s+, -1) AS word#36]
               +- Filter ((size(split(value#34, \s+, -1), true) > 0) AND isnotnull(split(value#34, \s+, -1)))
                  +- FileScan text [value#34] Batched: false, DataFilters: [(size(split(value#34, \s+, -1), true) > 0), isnotnull(split(value#34, \s+, -1))], Format: Text, Location: InMemoryFileIndex(1 paths)[hdfs://nameservice1/datos/README.md], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<value:string>




In [13]:
wordcount.cache()

DataFrame[word: string, count: bigint]

In [14]:
wordcount.is_cached

True

In [15]:
wordcount.count()

287

Palabras que aparecen 4 veces:

In [16]:
wordcount.filter(F.col("count") == 4).show()

+---------+-----+
|     word|count|
+---------+-----+
|      you|    4|
|     with|    4|
|      You|    4|
|   Please|    4|
|    build|    4|
|       an|    4|
|including|    4|
|       if|    4|
|     also|    4|
+---------+-----+



### Pandas y Spark

![](img/pandas_logo.png)

Al ser estructuras muy parecidas podemos cambiar de una a otra, teniendo en cuenta que si los datos son grandes no podemos trabajar en *pandas* (solo una máquina).

In [17]:
df = pd.DataFrame(np.random.randn(6, 4), columns=list("ABCD"))
df["date"] = pd.date_range("20130101", periods=6).astype("str")

In [18]:
df_spark = spark.createDataFrame(df)

In [19]:
df_spark.printSchema()

root
 |-- A: double (nullable = true)
 |-- B: double (nullable = true)
 |-- C: double (nullable = true)
 |-- D: double (nullable = true)
 |-- date: string (nullable = true)



In [20]:
df_spark.show()

[Stage 25:>                                                         (0 + 1) / 1]

+-------------------+--------------------+--------------------+--------------------+----------+
|                  A|                   B|                   C|                   D|      date|
+-------------------+--------------------+--------------------+--------------------+----------+
|0.06958738996280212|  0.6641715589312899|-0.12112091597372833|-0.11775471346519929|2013-01-01|
|  0.617620750992158|  0.1301849725253633| -0.1652513536806298|   1.148119065642691|2013-01-02|
|-0.8679318186254842|-0.20793660459756835|   1.569410440894088|  1.8004744302193758|2013-01-03|
| 0.7458519616043776| 0.17193205027370193|0.018212702030773967|  1.3137715761999191|2013-01-04|
| 0.6107490562176715|   0.557375834443788| -0.5926691041283254|  0.4707189305106173|2013-01-05|
|-0.6928839023046901|  0.7202227787423089|-0.16629415392971236|  0.9667002200740475|2013-01-06|
+-------------------+--------------------+--------------------+--------------------+----------+



In [21]:
nuevo_df = (
    df_spark.withColumn("nuevo", F.abs(F.col("A") - F.col("B")) / F.abs(F.col("D"))).select(
        "date", "nuevo"
    )
).toPandas()

In [22]:
type(nuevo_df)

pandas.core.frame.DataFrame

In [23]:
nuevo_df

,date,nuevo
0,2013-01-01,5.049345
1,2013-01-02,0.424552
2,2013-01-03,0.366567
3,2013-01-04,0.436849
4,2013-01-05,0.113387
5,2013-01-06,1.461784


### Lectura de CSV

![](img/csv.png)
&nbsp;   

Hemos visto como crear DataFrames leyendo texto con `spark.read` o como crearlos desde *pandas*. Pero la potencía de spark `DataFrames` es trabajar con datos estructurados. Es decir registros con columnas y cada columna de tipos diferentes. Veamos ahora como leer csv desde el hdfs:

In [24]:
diamonds = spark.read.text("/datos/diamonds.csv")

In [25]:
diamonds.show(4)

+--------------------+
|               value|
+--------------------+
|"carat","cut","co...|
|0.23,"Ideal","E",...|
|0.21,"Premium","E...|
|0.23,"Good","E","...|
+--------------------+
only showing top 4 rows



In [26]:
diamonds.take(4)

[Row(value='"carat","cut","color","clarity","depth","table","price","x","y","z"'),
 Row(value='0.23,"Ideal","E","SI2",61.5,55,326,3.95,3.98,2.43'),
 Row(value='0.21,"Premium","E","SI1",59.8,61,326,3.89,3.84,2.31'),
 Row(value='0.23,"Good","E","VS1",56.9,65,327,4.05,4.07,2.31')]

Usaremos `spark.read.csv`

In [27]:
diamonds = spark.read.csv("/datos/diamonds.csv")

In [28]:
diamonds.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)



In [29]:
diamonds.show(3)

+-----+-------+-----+-------+-----+-----+-----+----+----+----+
|  _c0|    _c1|  _c2|    _c3|  _c4|  _c5|  _c6| _c7| _c8| _c9|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
|carat|    cut|color|clarity|depth|table|price|   x|   y|   z|
| 0.23|  Ideal|    E|    SI2| 61.5|   55|  326|3.95|3.98|2.43|
| 0.21|Premium|    E|    SI1| 59.8|   61|  326|3.89|3.84|2.31|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
only showing top 3 rows



Parece que no ha utilizado la cabecera, pero es una opción:

In [30]:
diamonds = spark.read.options(header=True).csv("/datos/diamonds.csv")

In [31]:
diamonds.printSchema()

root
 |-- carat: string (nullable = true)
 |-- cut: string (nullable = true)
 |-- color: string (nullable = true)
 |-- clarity: string (nullable = true)
 |-- depth: string (nullable = true)
 |-- table: string (nullable = true)
 |-- price: string (nullable = true)
 |-- x: string (nullable = true)
 |-- y: string (nullable = true)
 |-- z: string (nullable = true)



In [32]:
diamonds.show(3)

+-----+-------+-----+-------+-----+-----+-----+----+----+----+
|carat|    cut|color|clarity|depth|table|price|   x|   y|   z|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
| 0.23|  Ideal|    E|    SI2| 61.5|   55|  326|3.95|3.98|2.43|
| 0.21|Premium|    E|    SI1| 59.8|   61|  326|3.89|3.84|2.31|
| 0.23|   Good|    E|    VS1| 56.9|   65|  327|4.05|4.07|2.31|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
only showing top 3 rows



Ya hemos arreglado las cabeceras, pero todos los campos son de tipo `string`. Por defecto será así, podemos pedir que infiera los tipos (aunque esto hará que tarde más la lectura):

In [33]:
diamonds = spark.read.options(header=True, inferSchema=True).csv("/datos/diamonds.csv")

In [34]:
diamonds.printSchema()

root
 |-- carat: double (nullable = true)
 |-- cut: string (nullable = true)
 |-- color: string (nullable = true)
 |-- clarity: string (nullable = true)
 |-- depth: double (nullable = true)
 |-- table: double (nullable = true)
 |-- price: integer (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- z: double (nullable = true)



In [35]:
diamonds.show(3)

+-----+-------+-----+-------+-----+-----+-----+----+----+----+
|carat|    cut|color|clarity|depth|table|price|   x|   y|   z|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
| 0.23|  Ideal|    E|    SI2| 61.5| 55.0|  326|3.95|3.98|2.43|
| 0.21|Premium|    E|    SI1| 59.8| 61.0|  326|3.89|3.84|2.31|
| 0.23|   Good|    E|    VS1| 56.9| 65.0|  327|4.05|4.07|2.31|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
only showing top 3 rows



In [36]:
filtrado = diamonds.filter("cut = 'Fair' ").filter(F.col("clarity") == "SI2")

In [37]:
filtrado.count()

466

In [38]:
filtrado.show(5)

+-----+----+-----+-------+-----+-----+-----+----+----+----+
|carat| cut|color|clarity|depth|table|price|   x|   y|   z|
+-----+----+-----+-------+-----+-----+-----+----+----+----+
| 0.86|Fair|    E|    SI2| 55.1| 69.0| 2757|6.45|6.33|3.52|
| 0.96|Fair|    F|    SI2| 66.3| 62.0| 2759|6.27|5.95|4.07|
| 0.91|Fair|    H|    SI2| 64.4| 57.0| 2763|6.11|6.09|3.93|
| 0.91|Fair|    H|    SI2| 65.7| 60.0| 2763|6.03|5.99|3.95|
| 0.98|Fair|    H|    SI2| 67.9| 60.0| 2777|6.05|5.97|4.08|
+-----+----+-----+-------+-----+-----+-----+----+----+----+
only showing top 5 rows



In [39]:
filtrado.limit(5).toPandas()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.86,Fair,E,SI2,55.1,69.0,2757,6.45,6.33,3.52
1,0.96,Fair,F,SI2,66.3,62.0,2759,6.27,5.95,4.07
2,0.91,Fair,H,SI2,64.4,57.0,2763,6.11,6.09,3.93
3,0.91,Fair,H,SI2,65.7,60.0,2763,6.03,5.99,3.95
4,0.98,Fair,H,SI2,67.9,60.0,2777,6.05,5.97,4.08


### Lectura desde Hive

![](img/hive.png)
&nbsp;  

Spark es comtaible con Hive, esto quiere decir que podemos leer cualquier tabla almacenada allí. Muchas empresas usan Hive como su *Data Warehouse* de big data. Y poder acceder desde spark a estos datos nos aisla de dónde están los datos, en qué formato están almacenados y cómo están comprimidos.

Con `spark.catalog` podemos acceder a la información del metastore y listar las bases de datos, tablas, ect.

In [40]:
spark.catalog.currentDatabase()

'default'

In [41]:
tablas = spark.catalog.listTables("jayuso")

In [42]:
tablas

[Table(name='agregado_2', catalog='spark_catalog', namespace=['jayuso'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='audiencias', catalog='spark_catalog', namespace=['jayuso'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='audiencias_large', catalog='spark_catalog', namespace=['jayuso'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='dns_agregado', catalog='spark_catalog', namespace=['jayuso'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='huge_table', catalog='spark_catalog', namespace=['jayuso'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='huge_table_v2', catalog='spark_catalog', namespace=['jayuso'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='passenger_role_type', catalog='spark_catalog', namespace=['jayuso'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='prueba', catalog='spark_catalog', namespace

In [43]:
pd.DataFrame(tablas)

,name,catalog,namespace,description,tableType,isTemporary
0,agregado_2,spark_catalog,[jayuso],None,MANAGED,False
1,audiencias,spark_catalog,[jayuso],None,MANAGED,False
2,audiencias_large,spark_catalog,[jayuso],None,MANAGED,False
3,dns_agregado,spark_catalog,[jayuso],None,MANAGED,False
4,huge_table,spark_catalog,[jayuso],None,MANAGED,False
5,huge_table_v2,spark_catalog,[jayuso],None,MANAGED,False
6,passenger_role_type,spark_catalog,[jayuso],None,MANAGED,False
7,prueba,spark_catalog,[jayuso],None,MANAGED,False
8,prueba5,spark_catalog,[jayuso],None,EXTERNAL,False
9,ratings,spark_catalog,[jayuso],None,MANAGED,False


Usamos `spark.table` para accerder a estas tablas, podemos hacerlo con `base_de_datos.nombre_de_la_tabla` o mejor, cambinado primero de base de datos:

In [44]:
spark.table("jayuso.restaurant")

DataFrame[url: string, address: string, address_line: string, name: string, outcode: string, postcode: string, rating: double, type_of_food: string]

In [45]:
spark.catalog.setCurrentDatabase("jayuso")

In [46]:
restaurantes = spark.table("restaurant")

In [47]:
restaurantes.printSchema()

root
 |-- url: string (nullable = true)
 |-- address: string (nullable = true)
 |-- address_line: string (nullable = true)
 |-- name: string (nullable = true)
 |-- outcode: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- type_of_food: string (nullable = true)



In [48]:
restaurantes.show(4)

[Stage 45:>                                                         (0 + 1) / 1]

+--------------------+-------------------+--------------+--------------------+-------+--------+------+------------+
|                 url|            address|  address_line|                name|outcode|postcode|rating|type_of_food|
+--------------------+-------------------+--------------+--------------------+-------+--------+------+------------+
|http://www.just-e...|    Stella Building|    Washington|        Albany Spice|   NE37|     1BH|   4.5|       Curry|
|http://www.just-e...|279 Manchester Road|West Yorkshire|           Albarakah|    HD4|     5AA|   4.5|       Curry|
|http://www.just-e...|  18 Sir Isaac Walk|    Colchester|             Albatta|    CO1|     1JJ|   5.0|    Lebanese|
|http://www.just-e...|    112 Gannow Lane|       Burnley|Alberto's Pizza &...|   BB12|     6QD|   5.5|       Kebab|
+--------------------+-------------------+--------------+--------------------+-------+--------+------+------------+
only showing top 4 rows



In [49]:
restaurantes.limit(4).toPandas()

,url,address,address_line,name,outcode,postcode,rating,type_of_food
0,http://www.just-eat.co.uk/restaurants-albany-s...,Stella Building,Washington,Albany Spice,NE37,1BH,4.5,Curry
1,http://www.just-eat.co.uk/restaurants-albaraka...,279 Manchester Road,West Yorkshire,Albarakah,HD4,5AA,4.5,Curry
2,http://www.just-eat.co.uk/restaurants-albatta-...,18 Sir Isaac Walk,Colchester,Albatta,CO1,1JJ,5.0,Lebanese
3,http://www.just-eat.co.uk/restaurants-albertos...,112 Gannow Lane,Burnley,Alberto's Pizza & Kebab House,BB12,6QD,5.5,Kebab


Una vez definido el `DataFrame` podemos trabajar como hasta ahora. Por ejemplo contar el número de restaurantes por tipo de cómida:

In [50]:
tipos_comida = restaurantes.groupBy("type_of_food").count().orderBy(F.desc("count"))

In [51]:
tipos_comida.show()

+--------------+-----+
|  type_of_food|count|
+--------------+-----+
|         Curry|  712|
|         Pizza|  387|
|       Chinese|  138|
|         Kebab|  127|
|  Fish & Chips|   99|
|      American|   83|
|       Turkish|   61|
|      Lebanese|   45|
|     Caribbean|   42|
|          Thai|   37|
|       Chicken|   37|
|       English|   24|
|       Burgers|   21|
|   Bangladeshi|   15|
|     Peri Peri|   13|
|Middle Eastern|   13|
|      Japanese|   12|
|         Grill|   11|
| Mediterranean|   11|
|       Mexican|   10|
+--------------+-----+
only showing top 20 rows



Igual que leemos también podemos escribir en hive:

In [52]:
tipos_comida.write.mode("overwrite").saveAsTable("jayuso.tipos_comida")

Otros ejemplos, buscar la hamburguesa con mayor puntuación:

In [53]:
mejor_burguer = (
    (restaurantes.filter("type_of_food = 'Burgers' ").orderBy(F.desc("rating"))).limit(1).toPandas()
)

In [54]:
mejor_burguer

,url,address,address_line,name,outcode,postcode,rating,type_of_food
0,http://www.just-eat.co.uk/restaurants-beastgou...,64 Torwood Street,Torquay,Beast Gourmet Burgers,TQ1,1DT,6.0,Burgers


Vamos ahora a buscar el mejor restaurante del tipo de comida con más restaurantes:

In [55]:
hay_mas = restaurantes.groupBy("type_of_food").count().orderBy(F.desc("count")).first()[0]

In [56]:
hay_mas

'Curry'

In [57]:
(restaurantes.filter(F.col("type_of_food") == hay_mas).orderBy(F.desc("rating"))).limit(
    1
).toPandas()

,url,address,address_line,name,outcode,postcode,rating,type_of_food
0,http://www.just-eat.co.uk/restaurants-amina-st...,5 Sutton Oak Corner,Birmingham,Amina,B74,2DH,6.0,Curry


O podemos hacerlo usando `join` con una sola acción:

In [58]:
(
    restaurantes.join(
        restaurantes.groupBy("type_of_food").count().orderBy(F.desc("count")).limit(1),
        "type_of_food",
        "leftsemi",
    ).orderBy(F.desc("rating"))
).limit(1).toPandas()

,type_of_food,url,address,address_line,name,outcode,postcode,rating
0,Curry,http://www.just-eat.co.uk/restaurants-amina-st...,5 Sutton Oak Corner,Birmingham,Amina,B74,2DH,6.0


La librería `DataFrame` también es comptabile con el lenguaje SQL, aunque por debajo también será un `DataFrame`:

In [59]:
spark.sql(
    """
        select x.* from restaurant x
        inner join (

            select type_of_food, count(*) as N from restaurant
            group by type_of_food
            order by N DESC
            limit 1

        ) y
        on x.type_of_food = y.type_of_food
        order by rating DESC
        limit 1
    """
).toPandas()

,url,address,address_line,name,outcode,postcode,rating,type_of_food
0,http://www.just-eat.co.uk/restaurants-amina-st...,5 Sutton Oak Corner,Birmingham,Amina,B74,2DH,6.0,Curry


In [60]:
restaurantes.unpersist()

DataFrame[url: string, address: string, address_line: string, name: string, outcode: string, postcode: string, rating: double, type_of_food: string]

####  Al finalizar, siempre hay que cerrar la conexión de spark:

In [61]:
spark.stop()